<a href="https://colab.research.google.com/github/Amulya-Rao-mj/GENAI-lab/blob/main/BasicHeathcareChatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch transformers accelerate gradio sentencepiece

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import gradio as gr

# 1. Zero-shot classifier (Ungated) to block non-health questions
print("Loading topic guardrail classifier...")
classifier = pipeline(
    "zero-shot-classification",
    model="cross-encoder/nli-deberta-v3-small"
)

# 2. Open-Access LLM (No API key or token needed)
print("Loading LLM model (Qwen 2.5 1.5B)...")
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

SYSTEM_PROMPT = """You are a basic healthcare information assistant.
Answer the user's question with short, factual health and medical information only.

IMPORTANT SAFETY RULES:
- Clarify that you are an AI, not a medical doctor.
- Do not provide formal diagnoses or prescribe medication.
- For emergency symptoms, instruct the user to call emergency services immediately."""

def is_health_related(user_query):
    candidate_labels = ["health or medical question", "general topic or off-topic conversation"]
    result = classifier(user_query, candidate_labels)
    return result['labels'][0] == "health or medical question"

def health_chatbot(message, history):
    # Step 1: Guardrail Check
    if not is_health_related(message):
        return "I am configured to answer medical and health-related questions only. Please ask a topic related to symptoms, wellness, or general healthcare."

    # Step 2: Format Messages
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": message}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Step 3: Generate Response
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.2,
        do_sample=True
    )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response

print("Launching interface...")
# Launch Local Web Interface directly inside Colab
gr.ChatInterface(
    fn=health_chatbot,
    title="Ungated Basic Health Chatbot",
    description="Ask general medical questions. No API keys required!"
).launch(inline=True, share=True)

Loading topic guardrail classifier...


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  568MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Launching interface...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b7546fddc5e290dfec.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
